In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "workspace")
dbutils.widgets.text("caminho_volume", "/Volumes/workspace/default/inputs/")

catalogo = dbutils.widgets.get("catalogo")
caminho_volume = dbutils.widgets.get("caminho_volume")
# Datas da consulta da cotação, no formato MM-DD-AAAA, que é o que a API exige.
dbutils.widgets.text("data_inicio", "")
# Em branco, o notebook usa os últimos 7 dias.
dbutils.widgets.text("data_fim", "")

In [0]:
#criação do schema da camada bronze
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.bronze")

DataFrame[]

In [0]:


#tabela de destino: 
arquivos = {
    "tb_movies_info":       ("movies_info_TMDB_IMDB.csv", True),
    "tb_movies_financials": ("movies_financials_IMDB_TMDB.csv", False),
    "tb_movies_metrics":    ("movies_metrics_IMDB_TMDB.csv", False),
    "tb_credits_and_tags":  ("credits_and_tags_IMDB_TMDB.csv", False),
    "tb_movies_reviews":    ("movies_reviews.csv", True),
}


for tabela, (arquivo, multiline) in arquivos.items():

    leitor = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", False)
    )
    if multiline:
 # multiLine: sinopses e comentários têm quebra de linha dentro das aspas
    # escape '"': aspas dentro do texto vêm duplicadas ("") e não com barra (\"), que é o padrão do Spark
        leitor = leitor.option("multiLine", True).option("escape", '"')

    df = leitor.load(f"{caminho_volume}/{arquivo}")

    # Momento da carga. Como a Bronze é append, a Silver usa essa coluna para
    # escolher o lote mais recente quando o mesmo filme aparecer mais de uma vez.
    df = df.withColumn("ingestion_datetime", F.current_timestamp())

    df.write.format("delta").mode("append").saveAsTable(f"{catalogo}.bronze.{tabela}")
    print(tabela, df.count())

tb_movies_info 106596
tb_movies_financials 106165
tb_movies_metrics 107364
tb_credits_and_tags 106320
tb_movies_reviews 32412


In [0]:
import requests
from datetime import date, timedelta

# Pega as datas dos widgets
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

# Se os widgets estiverem vazios, usa os últimos 7 dias
# (a API não tem cotação no fim de semana, então 7 dias garante que vem algum valor)
if data_inicio == "":
    data_inicio = (date.today() - timedelta(days=7)).strftime("%m-%d-%Y")
if data_fim == "":
    data_fim = date.today().strftime("%m-%d-%Y")

print("Buscando cotação de", data_inicio, "até", data_fim)

# URL da API do Banco Central com as datas
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

#chamada na API
resposta = requests.get(url)

# Transforma a resposta em dicionário e pega a lista de cotações
dados = resposta.json()
cotacoes = dados["value"]

# Cria o DataFrame com os campos do jeito que vieram da API
df_cotacao = spark.createDataFrame(cotacoes)

# Adiciona a data e hora da carga
df_cotacao = df_cotacao.withColumn("ingestion_datetime", F.current_timestamp())

# Salva na Bronze em Delta, em append
df_cotacao.write.format("delta").mode("append").saveAsTable(f"{catalogo}.bronze.tb_cotacao_dolar")

display(df_cotacao)

Buscando cotação de 09-29-2026 até 10-06-2026


cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.2198,2026-09-29 13:06:06.446305,2026-10-06T19:24:26.941Z
5.1803,2026-09-30 13:11:44.783262,2026-10-06T19:24:26.941Z
5.2073,2026-10-01 13:10:35.4469,2026-10-06T19:24:26.941Z
5.2232,2026-10-02 13:03:16.256632,2026-10-06T19:24:26.941Z
4.9853,2026-10-05 13:07:36.558602,2026-10-06T19:24:26.941Z
4.9692,2026-10-06 13:03:21.267287,2026-10-06T19:24:26.941Z
